# A tour of the rebuilt ProbPipe API

This notebook walks through the API on `dev/overhaul`, with one hierarchical model as the thread: the non-centered eight-schools model, whose parameters form the nested record `{population: {mu, tau}, groups: {theta_tilde}}`.
Each markdown cell says what the next cell shows, and each output is the library's own repr, declaration, table, or error message.
The last section lists what felt awkward while writing the notebook.

## Status

The design reference in `design/` is the target, and `dev/overhaul` implements most of it.
The pieces below are transitional, and each gets a one-line note where it first appears.

- **Imports.** The new operations are imported from their modules, as `tests/_ops.py` does. `probpipe.sample`, `probpipe.mean`, and the other top-level names are still the earlier implementations in `core/ops.py`. The next wave switches the exports, and the imports become `from probpipe import ...`.
- **Seeds.** `key=` arguments retire, and reproducible draws come from `workflow_run(seed=...)` (S1).
- **`expectation`.** It resolves through its own method registry today. It becomes the derived operation `mean(evaluate(f, d))`, so its controls and method names become `evaluate`'s and the registry retires (Z8). Every functional is an operation (Z10).
- **Labels.** A field view, a selection, and a marginal keep their parent's label, so `prior["beta"]` is labeled `prior` (Z5). The code still labels them by their path, and a fix is in progress.
- **Wave-2 review fixes in progress.** A cell that meets one of these keeps the behavior and marks it **Known bug**:
  1. two accesses of one batch element do not co-sample;
  2. an SG-MCMC or elliptical-slice chain on a factored prior starts outside the support;
  3. `probpipe.condition_on` of a factored joint at an upstream field runs MCMC where the exact slice applies;
  4. the empirical enumeration's exactness depends on argument order;
  5. `d[path]` at a regrouped node raises;
  6. `evaluate.with_options(method=...)` cannot name a rule;
  7. the moment matcher's `check` and its call disagree on dtypes;
  8. `from_distribution(..., check_support=False)` raises;
  9. `check` lists capability routes as approximate;
  10. a selection's marginal orders its fields by factor.
- **Open design questions**, marked **Open question** where they arise:
  1. the predictive checks take a sampling kernel, after which `GenerativeLikelihood` retires;
  2. a public capability for the density of a subset of observations;
  3. the top-level exports of `glm_likelihood` and `GaussianProcess`.
- **Docs.** The other notebooks in `docs/` are not updated yet, and a later pass rewrites them.

A cell marked **Finding** shows a behavior that differs from `design/` or surprised me and is not on the lists above.

## Setup

The operations come from their modules until the exports switch.
`show_report` prints a `CallReport` with one route per line.

In [1]:
import tempfile
import warnings
from pathlib import Path
from typing import Any

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

import probpipe
from probpipe import (
    BijectorTransformedDistribution, BootstrapDistribution, BootstrapReplicateDistribution,
    Dirichlet, DistributionBatch, EmpiricalDistribution, Gamma, HalfCauchy, KDEDistribution,
    MathematicalDomainError, MultivariateNormal, Normal, NumericArray, NumericArrayBatch,
    NumericArraySpec, NumericRecordBatch, Opaque, OutputSpec, Poisson, PyMCModel, Record,
    RecordBatch, RecordSpec, ResolutionError, StanModel, StudentT, SupportsCovariance,
    SupportsLogProb, SupportsMean, SupportsQuantile, SupportsSampling, SupportsUnnormalizedLogProb,
    SupportsVariance, bijector_for, converter_registry, function, iterate, positive,
    provenance_ancestors, real, replay_run, workflow_run,
)
from probpipe.distributions import (
    ConditionalDistribution, SupportsConditionalLogProb, SupportsConditionalSampling,
    SupportsFactors, SupportsMarginals,
)
from probpipe.families import PoissonFamily, glm_likelihood
from probpipe.operations import operation_registry

# The operations, from their modules until the top-level exports switch.
from probpipe.operations._condition import condition_on
from probpipe.operations._convert import convert
from probpipe.operations._density import log_prob, unnormalized_log_prob
from probpipe.operations._evaluate import evaluate
from probpipe.operations._marginal import factor, marginal
from probpipe.operations._moments import cov, expectation, mean, quantile, variance
from probpipe.operations._sample import sample

warnings.filterwarnings("ignore", message=r"Explicitly requested dtype float64")


def show_report(report):
    for route in report.routes:
        status = {True: "feasible", False: "declined", None: "unresolved"}[route.feasible]
        print(f"  {route.method_name:<34} {status:<9} exact={route.exact}  {route.description}")
    print("selected:", report.selected.method_name, "| exact:", report.selected.exact)

## 1. Values

A value is a tracked term: a representation together with a name and a spec.
`NumericArray` holds one array, whose shape is the event shape.

In [2]:
effect = NumericArray("effect", jnp.array([28.0, 8.0, -3.0]))
print(repr(effect))
print(effect.spec)
print(repr(effect + 1.0))

NumericArray('effect', Array([28.,  8., -3.], dtype=float32))
NumericArraySpec(shape=(3,), dtype=dtype('float32'), support=None)
Array([29.,  9., -2.], dtype=float32)


**Finding:** arithmetic on a `NumericArray` returns a bare `jax.Array`, while design III.1 returns a tracked term under a derived name.

A `Record` is a named tree of values, and a mapping-valued field becomes a subtree.
A subtree whose leaves are all numeric is promoted to a `NumericRecord`, so a school's record holds a numeric `data` subtree beside a string label.

In [3]:
school = Record("school_A", {"data": {"effect": 28.0, "se": 15.0}, "label": "A"})
print(type(school).__name__, school.spec)
print(list(school.keys()))
data = school.at_path("data")
print(type(data).__name__, data.name, data.spec)
print(repr(school["data/effect"]), repr(school["label"]))

Record RecordSpec(data=NumericRecordSpec(effect=(), se=()), label=None)
['data/effect', 'data/se', 'label']
NumericRecord data NumericRecordSpec(effect=(), se=())
Array(28., dtype=float32, weak_type=True) 'A'


**Finding:** `record[key]` returns the raw leaf, a `jax.Array` or the string, where design III.5 returns a view of the field's kind.

The eight-schools parameters form a `NumericRecord`.
It has the flat-vector interface, and `map` applies an elementwise function while keeping the structure.

In [4]:
theta = Record(
    "theta",
    {"population": {"mu": 4.0, "tau": 3.0}, "groups": {"theta_tilde": jnp.linspace(-1.0, 1.0, 8)}},
)
print(type(theta).__name__, theta.spec)
print(theta.vector_size, theta.to_vector())
print(repr(theta.map(jnp.abs)))
try:
    theta + theta
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

NumericRecord NumericRecordSpec(population=NumericRecordSpec(mu=(), tau=()), groups=NumericRecordSpec(theta_tilde=(8,)))
10 [ 4.          3.         -1.         -0.71428573 -0.42857143 -0.14285709
  0.1428572   0.42857146  0.71428585  1.        ]
NumericRecord(population=NumericRecord(mu=Array(4., dtype=float32, weak_type=True), tau=Array(3., dtype=float32, weak_type=True)), groups=NumericRecord(theta_tilde=array(shape=(8,))))
TypeError: unsupported operand type(s) for +: 'NumericRecord' and 'NumericRecord'


**Finding:** `+` between two records of one schema raises `TypeError`, while design III.5 gives `NumericRecord` the vector-space operators `+`, `-`, and scalar `*` and `/`.

`Opaque` wraps a value that no other kind admits, such as a tuple.
A mapping is refused, since the value layer reads a mapping as a subtree.

In [5]:
note = Opaque("source", ("Rubin (1981)", "SAT points"))
print(repr(note), note.spec, note.value)
try:
    Opaque("source", {"study": "Rubin (1981)"})
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

Opaque('source', ('Rubin (1981)', 'SAT points')) OpaqueSpec(meta=None) ('Rubin (1981)', 'SAT points')
TypeError: Opaque holds one unstructured value, and the value layer reads a mapping as a subtree rather than a leaf; wrap it as a Record, or as a non-mapping value


A `RecordBatch` stores records column by column on named levels.
Numeric and non-numeric fields sit side by side: a position gives a record, and a field gives its column.

In [6]:
y_obs = jnp.array([28.0, 8.0, -3.0, 7.0, -1.0, 1.0, 18.0, 12.0])
sigma = jnp.array([15.0, 10.0, 16.0, 11.0, 9.0, 11.0, 10.0, 18.0])
J = y_obs.shape[0]
schools_data = RecordBatch.stack(
    [Record("school", {"effect": y, "se": s, "label": label}) for y, s, label in zip(y_obs, sigma, "ABCDEFGH")],
    level_name="school",
    name="schools",
)
print(repr(schools_data), schools_data.element_spec)
print(repr(schools_data[2]), schools_data[2].name)
print(repr(schools_data.at_levels(school=2)))
print(repr(schools_data["label"]))
print(repr(schools_data["effect"]))

RecordBatch(name='schools', school=8) RecordSpec(effect=(), se=(), label=None)


Record(effect=Array(-3., dtype=float32), se=Array(16., dtype=float32), label='C') schools[school=2]
Record(effect=Array(-3., dtype=float32), se=Array(16., dtype=float32), label='C')
OpaqueBatch(name="schools['label']", school=8)
Array([28.,  8., -3.,  7., -1.,  1., 18., 12.], dtype=float32)


**Finding:** the numeric column comes back as a bare `jax.Array`, while the non-numeric column is a tracked `OpaqueBatch`. Design III.6 returns a tracked batch column for both.

The raw form is one call away for a law, whose `raw()` is its backend object.
A value's representation is read through `.value`, `.values`, `to_nested_dict()`, or indexing, and `sample.with_options(raw=True)` returns raw draws (Section 4).

In [7]:
print(effect.value)
print(school.to_nested_dict())
print(Normal("x", 0.0, 1.0).raw())
print("raw() on NumericArray, Record, RecordBatch:", hasattr(effect, "raw"), hasattr(school, "raw"), hasattr(schools_data, "raw"))

[28.  8. -3.]
{'data': {'effect': Array(28., dtype=float32, weak_type=True), 'se': Array(15., dtype=float32, weak_type=True)}, 'label': 'A'}
tfp.distributions.Normal("Normal", batch_shape=[], event_shape=[], dtype=float32)
raw() on NumericArray, Record, RecordBatch: False False False


**Finding:** the value kinds and the batches have no `raw()`, while design II.4 makes `raw()` the single access point of every tracked term.

## 2. Naming

A label names an object and has no mathematical meaning, while a component name addresses a part of what the object produces (C5).
A family's component defaults to its label, and `event_spec` names another.

In [8]:
prior_beta = Normal("prior", 0.0, 1.0, event_spec=OutputSpec(beta=None))
print(prior_beta.name, list(prior_beta.event_spec.components))
renamed = prior_beta.with_name("slope_prior")
print(renamed.name, list(renamed.event_spec.components))
print(prior_beta["beta"] is prior_beta)
print(prior_beta.event_spec)

prior ['beta']
slope_prior ['beta']
True
OutputSpec(_component_name='beta', _term_spec=NumericArraySpec(shape=(), dtype=dtype('float32'), support=real))


`with_name` changes the label alone, and indexing a whole-term law by its component returns the law itself.
The `OutputSpec` repr shows its private fields.

`with_path_names` renames nodes by their exact paths.
A target under the same parent renames in place, and a target under another parent moves the node, which promotes or demotes it.
All renames apply at once, so a swap is one call.

In [9]:
print("in place: ", list(school.with_path_names({"data/effect": "data/y"}).keys()))
print("promotion:", list(school.with_path_names({"data/effect": "effect"}).keys()))
print("demotion: ", list(school.with_path_names({"label": "meta/label"}).keys()))
print("swap:     ", school.with_path_names({"data/effect": "data/se", "data/se": "data/effect"}).to_nested_dict()["data"])
try:
    school.with_path_names({"effect": "y"})
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

in place:  ['data/y', 'data/se', 'label']
promotion: ['data/se', 'label', 'effect']
demotion:  ['data/effect', 'data/se', 'meta/label']
swap:      {'se': Array(28., dtype=float32, weak_type=True), 'effect': Array(15., dtype=float32, weak_type=True)}
KeyError: 'effect'


A key must be the exact path of a node, so the bare `effect` is refused; the `KeyError` names the key alone.

The same call renames a law's event.
Demoting the fields of a joint of three parametric priors builds the eight-schools prior with its nested record.
Demoting the component of a whole-term law is refused, since it would change the draw's kind.

In [10]:
flat_prior = Normal("mu", 0.0, 5.0) * HalfCauchy("tau", 0.0, 5.0) * Normal("theta_tilde", jnp.zeros(J), 1.0)
NEST = {"mu": "population/mu", "tau": "population/tau", "theta_tilde": "groups/theta_tilde"}
prior = flat_prior.with_path_names(NEST)
print(repr(flat_prior), list(flat_prior.event_spec.components))
print(repr(prior), list(prior.event_spec.components))
print(prior.event_spec.spec)
print("SupportsFactors:", isinstance(flat_prior, SupportsFactors), "->", isinstance(prior, SupportsFactors))
try:
    Normal("mu", 0.0, 5.0).with_path_names({"mu": "population/mu"})
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

FactoredDistribution(name='mu·tau·theta_tilde') ['mu', 'tau', 'theta_tilde']
_RenamedDistribution(name='mu·tau·theta_tilde', parent='mu·tau·theta_tilde') ['population', 'groups']
NumericRecordSpec(population=NumericRecordSpec(mu=NumericArraySpec(shape=(), dtype=dtype('float32'), support=real), tau=NumericArraySpec(shape=(), dtype=dtype('float32'), support=greater_than(0.0))), groups=NumericRecordSpec(theta_tilde=NumericArraySpec(shape=(8,), dtype=dtype('float32'), support=real)))
SupportsFactors: True -> False
ValueError: with_path_names() keeps the packaging, so the whole term's component 'mu' is renamed in place, not moved to 'population/mu'


**Finding:** the renamed joint is a private `_RenamedDistribution`, which no longer claims `SupportsFactors`. Design IV.1 renames a joint through its factors and returns a factored joint, and the docstring of `Distribution.with_path_names` promises `NotImplementedError` here (`probpipe/distributions/_distribution.py:496`). Sections 5 and 6 show the consequences.

## 3. Distributions

Each family derives its event declaration from its parameters: the component, shape, dtype, and support.
Capabilities are protocols, so `isinstance(d, SupportsMean)` reports whether a law implements a mean.

In [11]:
families = [
    Normal("mu", 0.0, 5.0),
    HalfCauchy("tau", 0.0, 5.0),
    Poisson("count", 3.0),
    Dirichlet("weights", jnp.ones(3)),
    MultivariateNormal("z", jnp.zeros(2), cov=jnp.array([[1.0, 0.5], [0.5, 2.0]])),
]
protocols = [SupportsSampling, SupportsLogProb, SupportsMean, SupportsVariance, SupportsCovariance, SupportsQuantile, SupportsMarginals]
pd.DataFrame(
    {
        "component": [next(iter(d.event_spec.components)) for d in families],
        "shape": [d.event_shape for d in families],
        "dtype": [str(d.dtype) for d in families],
        "support": [str(d.support) for d in families],
        **{p.__name__.removeprefix("Supports"): [isinstance(d, p) for d in families] for p in protocols},
    },
    index=[type(d).__name__ for d in families],
)

,component,shape,dtype,support,Sampling,LogProb,Mean,Variance,Covariance,Quantile,Marginals
Normal,mu,(),float32,real,True,True,True,True,True,True,False
HalfCauchy,tau,(),float32,greater_than(0.0),True,True,True,True,True,True,False
Poisson,count,(),float32,non_negative_integer,True,True,True,True,True,False,False
Dirichlet,weights,"(3,)",float32,simplex,True,True,True,True,True,False,False
MultivariateNormal,z,"(2,)",float32,real,True,True,True,True,True,True,False


A guard narrows a claim per instance.
`StudentT` claims `SupportsMean`, its guard decides existence once the degrees of freedom are concrete, and a mean known not to exist raises `MathematicalDomainError` rather than `ResolutionError` (II.7).
`HalfCauchy` claims a mean on the same terms.
Section 6 shows a guard that declines.

In [12]:
t = StudentT("t", 0.8, 0.0, 1.0)
print(isinstance(t, SupportsMean))
for law in (t, HalfCauchy("tau", 0.0, 5.0)):
    try:
        mean(law)
    except MathematicalDomainError as error:
        print(f"{type(error).__name__}: {error}")

True
MathematicalDomainError: the mean of 't' does not exist: it is finite only for degrees of freedom above one
MathematicalDomainError: the mean of 'tau' does not exist: E[X] is infinite


Parameters with more axes than one law needs give one law whose extra axes are event axes.
A vector `loc` gives independent coordinates, and a `MultivariateNormal` whose `loc` has shape `(4, 2)` gives four independent rows, so `log_prob` of one draw is a scalar in both cases.
Separate laws form a `DistributionBatch` (Section 8).

In [13]:
coords = Normal("theta_tilde", jnp.zeros(J), 1.0)
rows = MultivariateNormal("rows", jnp.zeros((4, 2)), cov=jnp.array([[1.0, 0.8], [0.8, 1.0]]))
print(repr(coords), coords.event_spec.spec)
print(repr(rows), rows.event_spec.spec)
with workflow_run(seed=0):
    draw = sample(rows)
print(draw.value)
print(repr(log_prob(coords, jnp.zeros(J))), repr(log_prob(rows, draw)))

Normal(name='theta_tilde', event_shape=(8,)) NumericArraySpec(shape=(8,), dtype=dtype('float32'), support=real)
MultivariateNormal(name='rows', event_shape=(4, 2)) NumericArraySpec(shape=(4, 2), dtype=dtype('float32'), support=real)


[[-0.9115058  -0.71506697]
 [-0.31258115 -0.93307924]
 [-0.25014585  0.15666881]
 [-0.56317055 -0.83777404]]


NumericArray('log_prob', Array(-7.3515086, dtype=float32)) NumericArray('log_prob', Array(-6.9956217, dtype=float32))


## 4. Operations on laws

The operations form a registry, and `describe` lists an operation's operands and its routes in selection order.

In [14]:
print([summary.name for summary in operation_registry.list()])
print(operation_registry.describe("mean"))

['convert', 'sample', 'condition_on', 'log_prob', 'unnormalized_log_prob', 'prob', 'unnormalized_prob', 'random_log_prob', 'random_unnormalized_log_prob', 'evaluate', 'inverse', 'log_det_jacobian', 'joint', 'marginal', 'factor', 'mixture', 'mean', 'variance', 'cov', 'quantile', 'expectation']
mean(d) — primitive
  The mean ``E[X]`` for ``X ~ d``, a value shaped like one draw.
  operands:
    d: DistributionSpec; read by the result rule
  routes, in selection order:
    closed_form (capability, exact; requires SupportsMean): membership in SupportsMean suffices
    monte_carlo (fallback, approximate): The law samples, and the moment is assumed to exist.


`sample` returns one tracked draw, or a batch on a level named `sample`, and `with_options(raw=True)` returns the representation instead.
Draws inside `workflow_run(seed=...)` are reproducible (Section 12).
**Transitional:** `key=` arguments retire, and the scope's seed fixes the draws (S1).

In [15]:
mu_prior = Normal("mu", 0.0, 5.0)
with workflow_run(seed=0):
    one = sample(mu_prior)
    many = sample(mu_prior, sample_shape=(5,))
    raw_many = sample.with_options(raw=True)(mu_prior, sample_shape=(5,))
    prior_draws = sample(prior, sample_shape=(3,))
print(repr(one))
print(repr(many), many.values)
print(type(raw_many).__name__, raw_many)
print(repr(prior_draws))
print(repr(prior_draws[0]))

NumericArray('sample', Array(-4.557529, dtype=float32))
NumericArrayBatch(batch_shape=(5,), levels=('sample',), event_shape=()) [-5.127493  -1.9288195 -0.6630897  6.3674555  3.6620731]
ArrayImpl [-14.860659   -3.721889   -6.036863    3.994918    6.0235777]
NumericRecordBatch(name='sample', sample=3)
NumericRecord(population=NumericRecord(mu=Array(3.6239264, dtype=float32), tau=Array(8.899201, dtype=float32)), groups=NumericRecord(theta_tilde=array(shape=(8,))))


**Finding:** every result is labeled by the operation alone (`sample`, `log_prob`, `mean`), so two means of different laws share the label `mean`, where II.4 derives a result's name from its inputs.

The functionals return values of the event's kind.
`cov` is over the flattened event, and `quantile` at several levels returns a batch on a level named `quantile`, the level axis leading.

In [16]:
z = families[-1]
print(repr(log_prob(z, jnp.zeros(2))))
print(repr(mean(z)), repr(variance(z)))
print(repr(cov(z)))
levels = quantile(z, jnp.array([0.05, 0.5, 0.95]))
print(repr(levels))
print(levels.values)
print(repr(quantile(z, 0.5)))

NumericArray('log_prob', Array(-2.1176848, dtype=float32))
NumericArray('mean', Array([0., 0.], dtype=float32)) NumericArray('variance', Array([1., 2.], dtype=float32))
NumericArray('cov', Array([[1. , 0.5],
       [0.5, 2. ]], dtype=float32))


NumericArrayBatch(batch_shape=(3,), levels=('quantile',), event_shape=(2,))
[[-1.6448536 -2.3261743]
 [ 0.         0.       ]
 [ 1.6448536  2.3261743]]


NumericArray('quantile', Array([0., 0.], dtype=float32))


**Finding:** `cov` returns a dense `NumericArray`, while design VI.5 returns a `LinOp` over the flattened draw.

`expectation(d, f)` integrates `f`, and `evaluate(f, d)` returns the pushforward law, which for a nonlinear map is an empirical law over the outputs.
**Transitional:** `expectation` resolves through its own method registry, whose methods are `exact` and `monte_carlo`. It becomes `mean(evaluate(f, d))`, with `evaluate`'s controls and method names (Z8).

In [17]:
@function
def square(x):
    return x**2


with workflow_run(seed=0):
    second_moment = expectation(mu_prior, square)
    pushforward = evaluate(square, mu_prior)
print(repr(second_moment), "(exact: 25)")
print(repr(pushforward), pushforward.event_spec.spec)
print(pushforward.provenance.metadata)
print(repr(mean(pushforward)))

NumericArray('expectation', Array(23.685886, dtype=float32)) (exact: 25)
EmpiricalDistribution(name='evaluate', num_atoms=128) NumericArraySpec(shape=(), dtype=dtype('float32'), support=None)
{'func': 'evaluate', 'route': 'evaluation_rules', 'exact': False}
NumericArray('mean', Array(29.23074, dtype=float32))


`check` runs the call up to the selection of a route without executing it, and returns a `CallReport`: the routes in selection order, the selected one, and the planned result.

In [18]:
report = mean.check(z)
print(report)
show_report(report)
show_report(expectation.check(mu_prior, square))

CallReport(routes=(MethodInfo(feasible=True, description='', pending=(), method_name='closed_form', exact=False),), selected=MethodInfo(feasible=True, description='', pending=(), method_name='closed_form', exact=True), deferred=(), result=OutputSpec(_component_name='z', _term_spec=NumericArraySpec(shape=(2,), dtype=dtype('float32'), support=real)), lifted=(), conversions=mappingproxy({}))
  closed_form                        feasible  exact=False  
selected: closed_form | exact: True
  methods (exact methods)            declined  exact=False  No feasible method for Normal with exact_only. Tried: exact: Normal has no exact expectation
  methods (approximate methods)      feasible  exact=False  
selected: methods/monte_carlo | exact: False


**Known bug:** `check` lists capability routes as approximate, so the `closed_form` route reads `exact=False` among the routes and `exact=True` once selected.